In [1]:
!pip install transformers

In [2]:
!pip install torch --quiet

In [7]:
!pip install huggingface_hub[hf_xet]

   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.9 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.9 MB 2.1 MB/s eta 0:00:02
   ---------------------------- ----------- 2.1/2.9 MB 4.9 MB/s eta 0:00:01
   ---------------------------------------  2.9/2.9 MB 4.1 MB/s eta 0:00:01
   ---------------------------------------  2.9/2.9 MB 4.1 MB/s eta 0:00:01
   ---------------------------------------- 2.9/2.9 MB 2.9 MB/s eta 0:00:00


In [3]:
import pandas as pd
from transformers import pipeline

In [16]:
import torch
import tqdm

In [18]:
# Detect GPU if available
device = 0 if torch.cuda.is_available() else -1
print("✅ Using GPU" if device == 0 else "⚠️ Using CPU (this will be slower)")

⚠️ Using CPU (this will be slower)


In [19]:
# Load the FinTwitBERT sentiment model
pipe = pipeline(
    "sentiment-analysis",
    model="StephanAkkerman/FinTwitBERT-sentiment",
    device=device
)

Device set to use cpu


In [20]:
# Load your dataset
df = pd.read_csv("Downloads/stock_tweets.csv")  # change to your actual filename
tweets = df["Tweet"].astype(str).tolist()  # your column name is "Tweet"

In [21]:
# Batch processing setup
batch_size = 64  # Adjust based on your GPU/CPU capacity
results = []
output_file = "your_dataset_with_sentiment.csv"

In [23]:
from tqdm import tqdm

In [24]:
# Process tweets in batches
for i in tqdm(range(0, len(tweets), batch_size)):
    batch = tweets[i:i+batch_size]
    preds = pipe(batch)
    batch_labels = [p["label"] for p in preds]
    results.extend(batch_labels)

    # Optional: save partial progress every 5000 tweets
    if i % 5000 == 0 and i > 0:
        temp_df = pd.DataFrame({
            "Tweet": tweets[:len(results)],
            "Sentiment": results
        })
        temp_df.to_csv(output_file, index=False)
        print(f"💾 Progress saved at {len(results)} tweets...")

# Add results back to your DataFrame
df["Sentiment"] = results
df.to_csv(output_file, index=False)

print(f"✅ Done! Full results saved to {output_file}")


 50%|█████████████████████████████████████▏                                     | 626/1263 [1:27:08<1:30:12,  8.50s/it]

💾 Progress saved at 40064 tweets...


 99%|███████████████████████████████████████████████████████████████████████████▎| 1251/1263 [2:58:47<01:46,  8.83s/it]

💾 Progress saved at 80064 tweets...


100%|████████████████████████████████████████████████████████████████████████████| 1263/1263 [3:00:52<00:00,  8.59s/it]


✅ Done! Full results saved to your_dataset_with_sentiment.csv
